# Learning module 4 — Check Appium readiness

**Small milestone:** inspect the automation setup without installing software, starting a server, or creating a device session.

The Appium server is the device-automation service, separate from our future agentic server and the TMDB API. It runs on Node.js. The XCUITest driver supplies iOS automation support. A Python Appium client is a separate dependency used by our proposed Python adapter; it is not the Appium server.

Read-only checks on **November 2, 2026 (America/Chicago)** found:

| Component | Observation | Meaning |
| --- | --- | --- |
| Node.js | `v20.17.0` on the checked PATH | Below the documented current Appium range. |
| npm | `10.8.3` | Meets the documented npm minimum; package installation has not been tested. |
| Appium command | Not found on PATH or in the inspected npm global location | We cannot run the CLI from this environment yet. This is not an exhaustive search of every possible installation. |
| XCUITest driver | No package manifest in the inspected Appium home | Driver registration remains unverified; the CLI is not available for its installed-driver check. |
| Python client | `appium` and `selenium` modules absent from the project's `.venv` | Our Python adapter dependencies are not installed in that interpreter. |
| Server status | Connection refused at `http://127.0.0.1:4723/status` | No server responded at this candidate endpoint during the check. |
| Xcode | `27.0`, build `27A266a` | Xcode is present; WebDriverAgent and actual session creation remain untested. |

The current Appium requirements list Node `^20.19.0 || ^22.12.0 || >=24.0.0` and npm `>=10`. Our Node version does not satisfy that range. [Official requirements](https://appium.io/docs/en/latest/quickstart/requirements/)

The iOS 27 target also requires a compatible server/driver combination. The current XCUITest matrix lists iOS 27 support from driver 11.1.2, maps driver 10.0.0–12.22.8 to Appium 3, and maps 13.0.0 onward to Appium 4. We will choose and pin a compatible combination during setup rather than independently install arbitrary latest versions. [XCUITest compatibility matrix](https://appium.github.io/appium-xcuitest-driver/latest/getting-started/system-requirements/)

The next cell reruns a small subset of these checks using **the notebook kernel's** interpreter, PATH, Appium home, and candidate server address. Its timestamp uses America/Chicago. It does not repair the setup.

## This notebook

Part **04 — Inspect environment and server readiness** of the learning sequence. [Notebook index](README.md).

Select the project's `.venv` Python kernel and run this notebook from top to bottom. Its Python setup is included locally; running another notebook in another kernel does not populate this one's variables or PATH. The supported working directories are the project root, `server/`, and `server/sample_scenarios/`.

**Execution scope:** Read-only checks and optional localhost status request.

Saved lesson outputs and embedded screenshots are historical snapshots preserved during the split on November 3, 2026. Newly included setup cells have no saved outputs. Rerunning an executable scenario creates fresh local artifacts; these continue living in `server/.tools/`, not beside the notebooks.


In [8]:
from datetime import datetime
import importlib.util
import json
import os
from pathlib import Path
from pprint import pprint
import shutil
import subprocess
import sys
import urllib.error
import urllib.request
from zoneinfo import ZoneInfo

node_path = shutil.which("node")
appium_home = Path(os.environ.get("APPIUM_HOME", str(Path.home() / ".appium")))
status_url = "http://127.0.0.1:4723/status"

readiness = {
    "checked_at": datetime.now(ZoneInfo("America/Chicago")).isoformat(timespec="seconds"),
    "kernel_python": sys.executable,
    "node_version": subprocess.check_output(
        [node_path, "--version"], text=True, timeout=10
    ).strip() if node_path else None,
    "appium_executable": shutil.which("appium"),
    "python_client_importable": importlib.util.find_spec("appium") is not None,
    "selenium_importable": importlib.util.find_spec("selenium") is not None,
    "appium_home": str(appium_home),
    "xcuitest_manifest_present": (
        appium_home / "node_modules/appium-xcuitest-driver/package.json"
    ).is_file(),
    "status_url": status_url,
}

try:
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    with opener.open(status_url, timeout=3) as response:
        value = json.load(response).get("value", {})
        readiness["server"] = {
            "http_status": response.status,
            "ready": value.get("ready"),
            "message": value.get("message"),
        }
except (urllib.error.URLError, TimeoutError, OSError, ValueError, AttributeError, TypeError) as error:
    readiness["server"] = {
        "status": "check_failed",
        "error_type": type(error).__name__,
        "http_status": getattr(error, "code", None),
        "detail": str(error),
    }

pprint(readiness, sort_dicts=False)


{'checked_at': '2026-11-02T20:56:22-06:00',
 'kernel_python': '/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/.venv/bin/python',
 'node_version': 'v24.22.2',
 'appium_executable': '/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/node_modules/.bin/appium',
 'python_client_importable': True,
 'selenium_importable': True,
 'appium_home': '/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/appium',
 'xcuitest_manifest_present': True,
 'status_url': 'http://127.0.0.1:4723/status',
 'server': {'status': 'check_failed',
            'error_type': 'URLError',
            'http_status': None,
            'detail': '<urlopen error [Errno 61] Connection refused>'}}


### Read the checks, then pause

- `shutil.which` tells us what commands this kernel can find on its PATH.
- `find_spec` checks whether a Python module is discoverable in this interpreter; it does not install or import the client.
- `xcuitest_manifest_present` checks one configured package location. It does not prove driver registration or compatibility. Once the CLI is available, inspect its installed-driver list.
- `GET /status` queries the candidate Appium endpoint. It does not create a session or control Movies. Even a ready server response would not prove that WebDriverAgent or our iOS session works. [Appium WebDriver API](https://appium.io/docs/en/latest/reference/api/webdriver/)

If your notebook output differs from the recorded findings, first compare the interpreter and PATH with the checked environment. `check_failed` means the status probe did not complete successfully, not necessarily that the server is unreachable. Connection-refused, timeout, HTTP error, malformed response, and permission-denied results mean different things; preserve the actual error instead of reporting every failure as “Appium not installed.”

This milestone's outcome is **setup work required**, not an application test failure. We have not evaluated Interstellar, the UI, or the backend.

**Next small setup milestone:** prepare a supported Node environment and verify its version. Then install a compatible Appium server/driver pair and the Python client in separate, inspectable steps. Only after readiness checks succeed will we create the first device session. Dependencies, lockfiles, server processes, and app state are unchanged by this lesson.

[Index](README.md) · [Previous lesson](03_app_profile.ipynb) · [Next lesson](05_local_node_runtime.ipynb)